# Test Set Synthetic Generation

Generate 10 synthetic call center conversations per language pair using OpenAI API.
- Positive: complete conversation with `<|im_end|>`
- Negative: truncated at 0.2-0.7 of last message, no `<|im_end|>`

In [21]:
import os
import json
import random
from openai import OpenAI
from tqdm import tqdm

os.environ['OPENAI_API_KEY'] = 'sk-proj-xxx'  # <-- paste your key here
client = OpenAI(api_key=os.environ.get('OPENAI_API_KEY'))

LANGUAGES = [
    'chinese-english',
    'chinese-malay',
    'chinese-tamil',
    'english-chinese',
    'english-malay',
    'english-tamil',
    'malay-chinese',
    'malay-english',
    'malay-tamil',
    'tamil-chinese',
    'tamil-english',
    'tamil-malay',
]

CONVOS_PER_LANG = 10
MODEL = 'gpt-4o-mini'

print(f'Languages: {len(LANGUAGES)}')
print(f'Conversations per language: {CONVOS_PER_LANG}')
print(f'Total conversations: {len(LANGUAGES) * CONVOS_PER_LANG}')

Languages: 12
Conversations per language: 10
Total conversations: 120


In [22]:
LANG_NAMES = {
    'chinese': 'Mandarin Chinese',
    'english': 'English',
    'malay': 'Malay (Bahasa Melayu)',
    'tamil': 'Tamil',
}

TOPICS = [
    'billing inquiry',
    'account update',
    'service complaint',
    'technical support',
    'insurance claim',
    'loan application',
    'appointment scheduling',
    'product return',
    'subscription cancellation',
    'payment issue',
    'address change',
    'password reset',
    'delivery tracking',
    'refund request',
    'new account opening',
]


def generate_conversation(lang_pair, topic, num_turns):
    lang1, lang2 = lang_pair.split('-')
    lang1_name = LANG_NAMES[lang1]
    lang2_name = LANG_NAMES[lang2]

    prompt = f"""Generate a realistic call center conversation between a customer (caller) and an agent.

Rules:
- The customer speaks primarily in {lang1_name}, occasionally mixing in {lang2_name} words/phrases (code-switching)
- The agent responds primarily in {lang1_name}, also mixing in {lang2_name} where natural
- Topic: {topic}
- EXACTLY {num_turns} turn(s) total (1 turn = 1 customer message + 1 agent response = 2 messages). So {num_turns} turn(s) = {num_turns * 2} messages.
- Make it sound natural, like a real Malaysian/Southeast Asian call center
- The conversation should reach a natural conclusion

Return ONLY a JSON array of messages, each with "role" (either "assistance" or "assistant") and "content" fields.
- First message role should be "assistance" (the customer)
- Alternate between "assistance" (customer) and "assistant" (agent)
- Do NOT include system messages

Example format:
[{{"role": "assistance", "content": "customer message..."}}, {{"role": "assistant", "content": "agent response..."}}]"""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0.9,
        max_tokens=2000,
    )

    text = response.choices[0].message.content.strip()
    # Strip markdown code block if present
    if text.startswith('```'):
        text = text.split('\n', 1)[1]
        text = text.rsplit('```', 1)[0]
    
    messages = json.loads(text)
    return messages


print('Ready')

Ready


In [23]:
from concurrent.futures import ThreadPoolExecutor, as_completed

tasks = []
for lang_pair in LANGUAGES:
    topics = random.sample(TOPICS, CONVOS_PER_LANG)
    for i, topic in enumerate(topics):
        num_turns = 1 if i < CONVOS_PER_LANG // 2 else 2
        tasks.append((lang_pair, topic, num_turns))

print(f'Total tasks: {len(tasks)}')

conversations = []
failed = 0

with ThreadPoolExecutor(max_workers=20) as executor:
    futures = {
        executor.submit(generate_conversation, lang, topic, turns): (lang, topic, turns)
        for lang, topic, turns in tasks
    }
    for future in tqdm(as_completed(futures), total=len(futures), desc='Generating'):
        lang, topic, turns = futures[future]
        try:
            messages = future.result()
            conversations.append({
                'language': lang,
                'topic': topic,
                'messages': messages,
            })
        except Exception as e:
            failed += 1
            print(f'  FAILED: {lang} | {topic} | {e}')

print(f'\nTotal conversations: {len(conversations)}')
print(f'Failed: {failed}')

Total tasks: 120


Generating:  73%|███████▎  | 88/120 [00:13<00:05,  5.65it/s]

  FAILED: tamil-chinese | password reset | Extra data: line 1 column 261 (char 260)


Generating: 100%|██████████| 120/120 [00:23<00:00,  5.07it/s]


Total conversations: 119
Failed: 1


In [24]:
# Preview some conversations
for conv in conversations[:3]:
    print(f"\n=== {conv['language']} | {conv['topic']} ===")
    for msg in conv['messages']:
        print(f"  [{msg['role']}] {msg['content'][:100]}...")


=== chinese-malay | loan application ===
  [assistance] 你好，我想了解一下关于贷款申请的流程。可以帮我吗？我需要借钱来买车， boleh tak?...
  [assistant] 当然可以！您可以告诉我您想申请的贷款金额和期限吗？这样我可以更好地帮助您。...

=== chinese-malay | new account opening ===
  [assistance] 你好，我想要开一个新账户，可以帮我吗？我不太确定需要什么文件。...
  [assistant] 你好！当然可以帮助你。开新账户通常需要身份证和住址证明。你有没有这些文件呢？...

=== chinese-english | account update ===
  [assistance] 你好，我想更新我的账户信息。可以帮我一下吗？我在这个 app 上找不到选项，太复杂了。...
  [assistant] 当然可以！请问您需要更新哪些信息呢？是地址还是电话号码？我们可以一起解决这个问题。...


## Build positive and negative samples

In [25]:
def format_chat_template(messages):
    parts = []
    for msg in messages:
        parts.append(f'<|im_start|>{msg["role"]}\n{msg["content"]}<|im_end|>')
    return '\n'.join(parts)



def build_positive(messages):
    """Complete conversation with trailing <|im_end|> removed.
    Text is complete - model should predict <|im_end|> next."""
    text = format_chat_template(messages)
    if text.endswith('<|im_end|>'):
        text = text[:-len('<|im_end|>')]
    return text.rstrip()


def build_negative(messages):
    """Truncated conversation, no <|im_end|> at end."""
    text = format_chat_template(messages)

    # Strip trailing <|im_end|>
    if text.endswith('<|im_end|>'):
        text = text[:-len('<|im_end|>')]

    # Find last <|im_start|> block and truncate within it
    last_im_start = text.rfind('<|im_start|>')
    if last_im_start >= 0:
        newline_after_role = text.find('\n', last_im_start)
        if newline_after_role >= 0:
            content_start = newline_after_role + 1
            content = text[content_start:]
            if len(content) > 10:
                keep_ratio = random.uniform(0.2, 0.7)
                keep_len = max(5, int(len(content) * keep_ratio))
                text = text[:content_start + keep_len]

    return text.rstrip()


# Build test samples
test_samples = []
for conv in conversations:
    messages = conv['messages']
    if not messages or len(messages) < 2:
        continue
    test_samples.append({
        'text': build_positive(messages),
        'label': 'positive',
        'language': conv['language'],
    })
    test_samples.append({
        'text': build_negative(messages),
        'label': 'negative',
        'language': conv['language'],
    })

print(f'Total test samples: {len(test_samples)}')
print(f'  Positive: {sum(1 for s in test_samples if s["label"] == "positive")}')
print(f'  Negative: {sum(1 for s in test_samples if s["label"] == "negative")}')

Total test samples: 238
  Positive: 119
  Negative: 119


In [26]:
# Verify samples look correct
for s in test_samples[:6]:
    text = s['text']
    ends_with_imend = text.endswith('<|im_end|>')
    print(f"[{s['label']:>8}] lang={s['language']:<20} ends_with_imend={ends_with_imend} len={len(text)}")
    print(f"  last 80 chars: ...{text[-80:]}")
    print()

[positive] lang=chinese-malay        ends_with_imend=False len=138
  last 80 chars: ...boleh tak?<|im_end|>
<|im_start|>assistant
当然可以！您可以告诉我您想申请的贷款金额和期限吗？这样我可以更好地帮助您。

[negative] lang=chinese-malay        ends_with_imend=False len=121
  last 80 chars: ...。可以帮我吗？我需要借钱来买车， boleh tak?<|im_end|>
<|im_start|>assistant
当然可以！您可以告诉我您想申请的贷款金额

[positive] lang=chinese-malay        ends_with_imend=False len=125
  last 80 chars: ...确定需要什么文件。<|im_end|>
<|im_start|>assistant
你好！当然可以帮助你。开新账户通常需要身份证和住址证明。你有没有这些文件呢？

[negative] lang=chinese-malay        ends_with_imend=False len=95
  last 80 chars: ...istance
你好，我想要开一个新账户，可以帮我吗？我不太确定需要什么文件。<|im_end|>
<|im_start|>assistant
你好！当然可以帮

[positive] lang=chinese-english      ends_with_imend=False len=140
  last 80 chars: ...，太复杂了。<|im_end|>
<|im_start|>assistant
当然可以！请问您需要更新哪些信息呢？是地址还是电话号码？我们可以一起解决这个问题。

[negative] lang=chinese-english      ends_with_imend=False len=118
  last 80 chars: ...以帮我一下吗？我在这个 app 上找不到选项，太复杂了。<|im_end|>
<|im_start|>assistant
当然可以！请问您

## Write to Parquet

In [27]:
import numpy as np
from transformers import AutoTokenizer
from chinidataset import ParquetWriter, StreamingDataset

TOKENIZER_NAME = 'Qwen/Qwen3-0.6B'
OUTPUT_DIR = './parquet-test-synthetic'

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)
im_end_id = tokenizer.convert_tokens_to_ids('<|im_end|>')
print(f'Tokenizer: {TOKENIZER_NAME}')
print(f'<|im_end|> token ID: {im_end_id}')

COLUMNS = {
    'input_ids': 'uint32[]',
    'position_ids': 'uint32[]',
    'attention_mask': 'uint32[]',
    'text': 'str',
    'raw_text': 'str',
}

with ParquetWriter(out=OUTPUT_DIR, columns=COLUMNS, exist_ok=True) as writer:
    for sample in tqdm(test_samples, desc='Writing test parquet'):
        outputs = tokenizer(sample['text'], add_special_tokens=False)
        ids = outputs['input_ids']
        if len(ids) == 0:
            continue

        meta = json.dumps({
            'label': sample['label'],
            'language': sample['language'],
        }, ensure_ascii=False)

        writer.write({
            'input_ids': np.array(ids, dtype=np.uint32),
            'position_ids': np.arange(len(ids), dtype=np.uint32),
            'attention_mask': np.array([len(ids)], dtype=np.uint32),
            'text': meta,
            'raw_text': sample['text'],
        })

print(f'\nWritten to {OUTPUT_DIR}')

# Verify
ds = StreamingDataset(local=OUTPUT_DIR)
print(f'Parquet samples: {len(ds)}')
row = ds[0]
print(f'Keys: {list(row.keys())}')
print(f'input_ids shape: {row["input_ids"].shape}')
print(f'text: {row["text"]}')

Directory /root/small-ablation/turn-detector/parquet-test-synthetic exists; removing contents.


Tokenizer: Qwen/Qwen3-0.6B
<|im_end|> token ID: 151645


Writing test parquet: 100%|██████████| 238/238 [00:00<00:00, 6359.70it/s]


Written to ./parquet-test-synthetic
Parquet samples: 238
Keys: ['input_ids', 'position_ids', 'attention_mask', 'text', 'raw_text']
input_ids shape: (54,)
text: {"label": "positive", "language": "chinese-malay"}


## Upload to S3

In [28]:
import subprocess

os.environ['AWS_ACCESS_KEY_ID'] = ''  # <-- paste your key here
os.environ['AWS_SECRET_ACCESS_KEY'] = ''  # <-- paste your key here

S3_BUCKET = 'aies-research-datasets'
S3_PREFIX = 'call-center-language-switching'

cmd = f'aws s3 sync {OUTPUT_DIR} s3://{S3_BUCKET}/{S3_PREFIX}/parquet-test-synthetic/'
print(f'Running: {cmd}')
result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(f'ERROR: {result.stderr}')
else:
    print(f'Done: s3://{S3_BUCKET}/{S3_PREFIX}/parquet-test-synthetic/')

Running: aws s3 sync ./parquet-test-synthetic s3://aies-research-datasets/call-center-language-switching/parquet-test-synthetic/
Completed 721 Bytes/274.0 KiB (1.2 KiB/s) with 2 file(s) remaining
upload: parquet-test-synthetic/index.json to s3://aies-research-datasets/call-center-language-switching/parquet-test-synthetic/index.json
Completed 721 Bytes/274.0 KiB (1.2 KiB/s) with 1 file(s) remaining
Completed 274.0 KiB/274.0 KiB (299.2 KiB/s) with 1 file(s) remaining
upload: parquet-test-synthetic/shard.00000.parquet to s3://aies-research-datasets/call-center-language-switching/parquet-test-synthetic/shard.00000.parquet

Done: s3://aies-research-datasets/call-center-language-switching/parquet-test-synthetic/


In [29]:
import pandas as pd                                                                                                                                                                       
df = pd.read_parquet('./parquet-test-synthetic/shard.00000.parquet')
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
df.head(10)  

,input_ids,position_ids,attention_mask,text,raw_text
0,"[151644, 395, 3924, 198, 108386, 3837, 104100, 110050, 101888, 101983, 101915, 9370, 102054, 1773, 73670, 108965, 101037, 11319, 35946, 85106, 116146, 36407, 113930, 3837, 708, 73790, 18116, 30, 151645, 198, 151644, 77091, 198, 103942, 73670, 6313, 107952, 106525, 87026, 99172, 101915, 9370, 101983, 80094, 33108, 105362, 101037, 11319, 99654, 109944, 105344, 100364, 87026, 1773]","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53]",[54],"{""label"": ""positive"", ""language"": ""chinese-malay""}",<|im_start|>assistance\n你好，我想了解一下关于贷款申请的流程。可以帮我吗？我需要借钱来买车， boleh tak?<|im_end|>\n<|im_start|>assistant\n当然可以！您可以告诉我您想申请的贷款金额和期限吗？这样我可以更好地帮助您。
1,"[151644, 395, 3924, 198, 108386, 3837, 104100, 110050, 101888, 101983, 101915, 9370, 102054, 1773, 73670, 108965, 101037, 11319, 35946, 85106, 116146, 36407, 113930, 3837, 708, 73790, 18116, 30, 151645, 198, 151644, 77091, 198, 103942, 73670, 6313, 107952, 106525, 87026, 99172, 101915, 9370, 101983, 80094]","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43]",[44],"{""label"": ""negative"", ""language"": ""chinese-malay""}",<|im_start|>assistance\n你好，我想了解一下关于贷款申请的流程。可以帮我吗？我需要借钱来买车， boleh tak?<|im_end|>\n<|im_start|>assistant\n当然可以！您可以告诉我您想申请的贷款金额
2,"[151644, 395, 3924, 198, 108386, 3837, 35946, 103945, 29767, 46944, 16628, 104827, 3837, 73670, 108965, 101037, 11319, 101553, 99222, 60610, 85106, 99245, 26898, 1773, 151645, 198, 151644, 77091, 198, 108386, 6313, 103942, 111728, 56568, 1773, 29767, 16628, 104827, 102119, 85106, 105765, 33108, 114870, 104022, 1773, 56568, 104710, 100001, 26898, 101036, 11319]","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50]",[51],"{""label"": ""positive"", ""language"": ""chinese-malay""}",<|im_start|>assistance\n你好，我想要开一个新账户，可以帮我吗？我不太确定需要什么文件。<|im_end|>\n<|im_start|>assistant\n你好！当然可以帮助你。开新账户通常需要身份证和住址证明。你有没有这些文件呢？
3,"[151644, 395, 3924, 198, 108386, 3837, 35946, 103945, 29767, 46944, 16628, 104827, 3837, 73670, 108965, 101037, 11319, 101553, 99222, 60610, 85106, 99245, 26898, 1773, 151645, 198, 151644, 77091, 198, 108386, 6313, 103942, 73670, 99663]","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33]",[34],"{""label"": ""negative"", ""language"": ""chinese-malay""}",<|im_start|>assistance\n你好，我想要开一个新账户，可以帮我吗？我不太确定需要什么文件。<|im_end|>\n<|im_start|>assistant\n你好！当然可以帮
4,"[151644, 395, 3924, 198, 108386, 3837, 104100, 50007, 97611, 104827, 27369, 1773, 73670, 108965, 100158, 101037, 11319, 35946, 104596, 906, 64118, 106486, 109487, 3837, 99222, 102181, 34187, 1773, 151645, 198, 151644, 77091, 198, 103942, 73670, 6313, 109194, 87026, 85106, 50007, 102224, 27369, 101036, 11319, 20412, 46477, 99998, 87805, 107154, 11319, 105773, 100018, 100638, 105073, 1773]","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54]",[55],"{""label"": ""positive"", ""language"": ""chinese-english""}",<|im_start|>assistance\n你好，我想更新我的账户信息。可以帮我一下吗？我在这个 app 上找不到选项，太复杂了。<|im_end|>\n<|im_start|>assistant\n当然可以！请问您需要更新哪些信息呢？是地址还是电话号码？我们可以一起解决这个问题。
5,"[151644, 395, 3924, 198, 108386, 3837, 104100, 50007, 97611, 104827, 27369, 1773, 73670, 108965, 100158, 101037, 11319, 35946, 104596, 906, 64118, 106486, 109487, 3837, 99222, 102181, 34187, 1773, 151645, 198, 151644, 77091, 198, 103942, 73670, 6313, 109194, 87026, 85106, 50007, 102224, 27369, 101036, 11319, 20412]","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 1

In [34]:
print(os.environ['HF_TOKEN'])

hf_xxx


In [48]:
print(HF_TOKEN)

hf_xxx


In [51]:
import os
from huggingface_hub import HfApi

HF_TOKEN = ""
REPO_ID = 'Scicom-intl/turn-detector-Qwen3-0.6B-TEST'
 
FOLDERS = {                                                                                                                                                                               
      'test': './parquet-test-synthetic',       
  }                                                                                                                                                                                         
   
 
api = HfApi(token=HF_TOKEN)
print('Ready')

Ready


In [52]:
api.create_repo(repo_id=REPO_ID, repo_type='dataset', exist_ok=True, private=False)
 
for folder_name, local_path in FOLDERS.items():
    print(f'\nUploading {folder_name} from {local_path}')
    api.upload_folder(
        folder_path=local_path,
        path_in_repo=folder_name,
        repo_id=REPO_ID,
        repo_type='dataset',
        commit_message=f'upload {folder_name}',
    )
    print(f'  Done')
 
print(f'\nAll done: https://huggingface.co/datasets/{REPO_ID}')


Uploading test from ./parquet-test-synthetic


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

  Done

All done: https://huggingface.co/datasets/Scicom-intl/turn-detector-Qwen3-0.6B-TEST
